# GCI World 2026 - v7: extra_trees in every member (on top of v6's learned-expectation features)
**Idea (new, training procedure - not a new feature).** The target-encoding features (q / product keys) are noisy, and flexible trees overfit them: more leaves or fewer rows per leaf made held-out AUC clearly worse. Randomised split thresholds (`extra_trees=True`) regularise against exactly this.
**Evidence (LightGBM, product-key one-hot features, 5-fold held-out, nested encodings, fast settings):** core one-hot 0.76987 -> 0.77065 (+0.00078); the same with another model seed and inner partition 0.76960 -> 0.77037 (+0.00077); all-families one-hot 0.76966 -> 0.77105 (+0.00139).
Flat around the default: nearby settings gained +0.0003 to +0.0008; more capacity (31 leaves) lost; 900 rounds lost (0.76942), so keep the rounds unchanged. NOT yet verified: the monotone member, the shallow member, production learning rates, and the effect inside the full blend - this notebook tests exactly that.
**What it does:** retrains the same five members with the SAME weights and the SAME rounds, `extra_trees=True` added, and compares fold by fold with your **v6 members** (Drive folder `gci_cache_v6`, written by the v6 notebook). The v6 residual features are reused from that folder (no recomputation).
**Rule set in advance:** submit a v7 file only if its 'minus v6' mean is >= +0.0002 AND positive on >= 4 of 5 folds. Otherwise keep v6_B (0.778).
Runtime about 1.2 hours on Colab CPU (extra_trees trees are slightly faster to build).

In [ ]:
# cell 1 - setup
import os, sys, glob, time, warnings; warnings.filterwarnings("ignore")
try:
    from google.colab import drive; drive.mount("/content/drive")
except Exception as e: print("Drive not mounted (ok if files are in /content):", e)
import numpy as np, pandas as pd, lightgbm as lgb
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
SMOKE = os.environ.get("SMOKE") == "1"                   # SMOKE=1 -> tiny quick test of the code path only
def find(name, roots=("/content", "/content/drive/MyDrive", os.getcwd())):
    for r in roots:
        for pat in (name, "*/"+name, "*/*/"+name, "*/*/*/"+name, "*/*/*/*/"+name):
            hit = glob.glob(os.path.join(r, pat))
            if hit: return hit[0]
    return None
TRAIN_PATH, TEST_PATH, SAMPLE_PATH = find("train.csv"), find("test.csv"), find("sample_submission.csv")
assert TRAIN_PATH and TEST_PATH and SAMPLE_PATH, "Could not find train.csv / test.csv / sample_submission.csv - upload them to /content or put them in Drive"
HGB_PATH = find("GCI_hgb_product_test.npy"); HGB_OOF = find("GCI_hgb_product_oof.npy")
base_dir = "/content/drive/MyDrive" if os.path.isdir("/content/drive/MyDrive") else "/content"
CACHE_DIR = os.path.join(base_dir, "gci_cache_v7") if not SMOKE else "/tmp/gci_cache_smoke7"; OUT_DIR = os.path.join(base_dir, "gci_output_v7") if not SMOKE else "/tmp/gci_out_smoke7"
V6_CACHE = os.path.join(base_dir, "gci_cache_v6")      # members + residual features written by the v6 notebook
os.makedirs(CACHE_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
THREADS = os.cpu_count() or 1
print("train:", TRAIN_PATH, "\ntest:", TEST_PATH, "\nsample:", SAMPLE_PATH, "\nhgb oof:", HGB_OOF, "\nhgb npy:", HGB_PATH, "\ncache:", CACHE_DIR, "| threads:", THREADS)

In [ ]:
# cell 2 - features, q and decoded-product keys
CATS = ["NAME_CONTRACT_TYPE","CODE_GENDER","NAME_INCOME_TYPE","NAME_EDUCATION_TYPE","NAME_FAMILY_STATUS","NAME_HOUSING_TYPE","OCCUPATION_TYPE","ORGANIZATION_TYPE"]
tr = pd.read_csv(TRAIN_PATH); te = pd.read_csv(TEST_PATH); sample = pd.read_csv(SAMPLE_PATH)
y = tr["TARGET"].values; ntr = len(tr)
al = pd.concat([tr.drop(columns=["TARGET"]), te], ignore_index=True)
def new_cols(d): return pd.DataFrame(d, index=al.index)
def base(al):
    X = al.drop(columns=["SK_ID_CURR"]).copy()
    for c in CATS: X[c] = X[c].astype("category").cat.codes.replace(-1, np.nan).astype(float)
    return X
def f_clean(al, X):
    d = {"DE_SENT": (al.DAYS_EMPLOYED == 365243).astype(int), "DAYS_EMPLOYED_CLEAN": al.DAYS_EMPLOYED.where(al.DAYS_EMPLOYED != 365243),
         "ORG_XNA": (al.ORGANIZATION_TYPE == "XNA").astype(int), "CAR_AGE_OUT": (al.OWN_CAR_AGE >= 60).astype(int), "MISS_CNT": al.isna().sum(axis=1)}
    for c in ["EXT_SOURCE_1","EXT_SOURCE_3","OWN_CAR_AGE","AMT_REQ_CREDIT_BUREAU_YEAR"]: d[c + "_NA"] = al[c].isna().astype(int)
    return pd.concat([X, new_cols(d)], axis=1)
def f_ext(al, X):
    e = al[["EXT_SOURCE_1","EXT_SOURCE_2","EXT_SOURCE_3"]]; d = {}
    d["EXT_MEAN"] = e.mean(axis=1); d["EXT_MIN"] = e.min(axis=1); d["EXT_MAX"] = e.max(axis=1); d["EXT_STD"] = e.std(axis=1)
    d["EXT_RANGE"] = d["EXT_MAX"] - d["EXT_MIN"]; d["EXT_NOBS"] = e.notna().sum(axis=1); d["EXT_PROD"] = e.prod(axis=1)
    d["EXT_W"] = (al.EXT_SOURCE_1*2 + al.EXT_SOURCE_2*3 + al.EXT_SOURCE_3*4) / 9
    d["E12"] = al.EXT_SOURCE_1*al.EXT_SOURCE_2; d["E13"] = al.EXT_SOURCE_1*al.EXT_SOURCE_3; d["E23"] = al.EXT_SOURCE_2*al.EXT_SOURCE_3
    d["D12"] = al.EXT_SOURCE_1-al.EXT_SOURCE_2; d["D13"] = al.EXT_SOURCE_1-al.EXT_SOURCE_3; d["D23"] = al.EXT_SOURCE_2-al.EXT_SOURCE_3
    return pd.concat([X, new_cols(d)], axis=1)
def f_fin(al, X):
    inc, cr, an, gp = al.AMT_INCOME_TOTAL, al.AMT_CREDIT, al.AMT_ANNUITY, al.AMT_GOODS_PRICE
    d = {"CREDIT_INC": cr/inc, "ANN_INC": an/inc, "ANN_CREDIT": an/cr, "TERM": cr/an, "GOODS_CREDIT": gp/cr, "CREDIT_MINUS_GOODS": cr-gp, "GOODS_INC": gp/inc,
         "INC_PER_FAM": inc/al.CNT_FAM_MEMBERS, "CREDIT_PER_FAM": cr/al.CNT_FAM_MEMBERS, "ANN_PER_FAM": an/al.CNT_FAM_MEMBERS}
    for c in ["AMT_INCOME_TOTAL","AMT_CREDIT","AMT_ANNUITY","AMT_GOODS_PRICE"]: d["LOG_" + c] = np.log1p(al[c])
    return pd.concat([X, new_cols(d)], axis=1)
def f_age(al, X):
    age = -al.DAYS_BIRTH; de = al.DAYS_EMPLOYED.where(al.DAYS_EMPLOYED != 365243)
    d = {"AGE": age/365.25, "EMP_AGE": de/al.DAYS_BIRTH, "EMP_YEARS": -de/365.25, "REG_AGE": al.DAYS_REGISTRATION/al.DAYS_BIRTH, "ID_AGE": al.DAYS_ID_PUBLISH/al.DAYS_BIRTH,
         "PHONE_AGE": al.DAYS_LAST_PHONE_CHANGE/al.DAYS_BIRTH, "CAR_AGE_RATIO": al.OWN_CAR_AGE/(age/365.25), "CREDIT_PER_AGE": al.AMT_CREDIT/(age/365.25), "INC_PER_AGE": al.AMT_INCOME_TOTAL/(age/365.25)}
    return pd.concat([X, new_cols(d)], axis=1)
def f_bureau(al, X):
    b = [c for c in al.columns if c.startswith("AMT_REQ")]
    d = {"BUR_SUM": al[b].sum(axis=1, min_count=1), "BUR_RECENT": al[["AMT_REQ_CREDIT_BUREAU_HOUR","AMT_REQ_CREDIT_BUREAU_MON"]].sum(axis=1, min_count=1),
         "BUR_MON_YEAR": al.AMT_REQ_CREDIT_BUREAU_MON/(al.AMT_REQ_CREDIT_BUREAU_YEAR+1)}
    return pd.concat([X, new_cols(d)], axis=1)
def f_count(al, X):
    d = {"CNT_" + c: al[c].map(al[c].value_counts()) for c in CATS + ["REGION_POPULATION_RELATIVE","DAYS_BIRTH","AMT_CREDIT","AMT_GOODS_PRICE","AMT_ANNUITY","AMT_INCOME_TOTAL"]}
    return pd.concat([X, new_cols(d)], axis=1)
def f_group(al, X):
    tmp = al.copy(); tmp["CREDIT_INC"] = al.AMT_CREDIT/al.AMT_INCOME_TOTAL; d = {}
    for k in ["ORGANIZATION_TYPE","OCCUPATION_TYPE","NAME_INCOME_TYPE","NAME_EDUCATION_TYPE","REGION_POPULATION_RELATIVE"]:
        for v in ["AMT_INCOME_TOTAL","AMT_CREDIT","EXT_SOURCE_1","EXT_SOURCE_2","EXT_SOURCE_3","DAYS_BIRTH","CREDIT_INC"]:
            d[f"G_{k[:6]}_{v[-6:]}_DIFF"] = tmp[v] - tmp.groupby(k)[v].transform("mean")
    return pd.concat([X, new_cols(d)], axis=1)
def f_kdec(al, X):    # credit = goods * (1 + 0.0132*k) for ~90% of cash loans with credit > goods -> integer k (insurance/tariff tier); NaN where another tariff family applies
    C, G = al.AMT_CREDIT, al.AMT_GOODS_PRICE; cash = al.NAME_CONTRACT_TYPE == "Cash loans"; kf = (C / G - 1) / 0.0132; k = kf.round(); ok = cash & (C > G) & ((kf - k).abs() < 0.1)
    K = pd.Series(np.where(ok, k, np.where(cash & (C == G), 0.0, np.nan)), index=al.index)
    return pd.concat([X, new_cols({"K_DEC": K, "CG_RATIO": C / G})], axis=1)
FAMS = {"kdec": f_kdec, "clean": f_clean, "ext": f_ext, "fin": f_fin, "age": f_age, "bureau": f_bureau, "count": f_count, "group": f_group}
def build(fams):
    X = base(al)
    for f in fams: X = FAMS[f](al, X)
    X = X.replace([np.inf, -np.inf], np.nan)
    return X.iloc[:ntr].reset_index(drop=True), X.iloc[ntr:].reset_index(drop=True)
CORE = ["clean", "fin", "age", "bureau"]; ALLF = ["clean", "ext", "fin", "age", "bureau", "count", "group"]
Xc, Xct = build(CORE); Xa, Xat = build(ALLF)                              # members WITHOUT k
Xck, Xckt = build(CORE + ["kdec"]); Xak, Xakt = build(ALLF + ["kdec"])   # members WITH decoded k (v2 members)
# ---- NEW (v6): learned-expectation residual features (label-free; fitted on train+test together; out-of-fold predictions) ----
from sklearn.ensemble import HistGradientBoostingRegressor as HR
from sklearn.model_selection import KFold
RES_PATH = os.path.join(V6_CACHE, "resid16.pkl") if os.path.exists(os.path.join(V6_CACHE, "resid16.pkl")) else os.path.join(CACHE_DIR, "resid16.pkl")
def compute_resid():
    raw = al.drop(columns=["SK_ID_CURR"]).copy()
    for c in CATS: raw[c] = raw[c].astype("category").cat.codes.replace(-1, np.nan).astype(float)
    raw["DAYS_EMPLOYED"] = raw.DAYS_EMPLOYED.where(raw.DAYS_EMPLOYED != 365243)
    demo = [c for c in raw.columns if c.startswith("EXT_SOURCE") or c.startswith("AMT_")]
    T = {"EXT1": (raw.EXT_SOURCE_1, ["EXT_SOURCE_1"]), "EXT2": (raw.EXT_SOURCE_2, ["EXT_SOURCE_2"]), "EXT3": (raw.EXT_SOURCE_3, ["EXT_SOURCE_3"]),
         "LOGINC": (np.log(raw.AMT_INCOME_TOTAL), ["AMT_INCOME_TOTAL"]), "LOGCR": (np.log(raw.AMT_CREDIT), ["AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"]),
         "EMP": (raw.DAYS_EMPLOYED, ["DAYS_EMPLOYED"]), "REG": (raw.DAYS_REGISTRATION, ["DAYS_REGISTRATION"]), "AGE": (raw.DAYS_BIRTH, ["DAYS_BIRTH"]),
         "IDP": (raw.DAYS_ID_PUBLISH, ["DAYS_ID_PUBLISH"]), "PHONE": (raw.DAYS_LAST_PHONE_CHANGE, ["DAYS_LAST_PHONE_CHANGE"]), "REGPOP": (raw.REGION_POPULATION_RELATIVE, ["REGION_POPULATION_RELATIVE"]),
         "FAM": (raw.CNT_FAM_MEMBERS, ["CNT_FAM_MEMBERS"]), "CAR": (raw.OWN_CAR_AGE, ["OWN_CAR_AGE"]),
         "EXT1D": (raw.EXT_SOURCE_1, demo), "EXT2D": (raw.EXT_SOURCE_2, demo), "EXT3D": (raw.EXT_SOURCE_3, demo)}   # *D = expected from demographics only (no EXT, no AMT columns)
    if SMOKE: T = dict(list(T.items())[:2])
    out = {}; t0 = time.time()
    for nm, (yt, drop_cols) in T.items():
        Xp = raw.drop(columns=drop_cols); pred = np.zeros(len(raw)); obs = yt.notna().values
        for tri, tei in KFold(2 if SMOKE else 5, shuffle=True, random_state=0).split(Xp):
            m = HR(max_iter=5 if SMOKE else 120, learning_rate=0.1, max_leaf_nodes=31, min_samples_leaf=100, random_state=0).fit(Xp.iloc[tri][obs[tri]], yt.iloc[tri][obs[tri]]); pred[tei] = m.predict(Xp.iloc[tei])
        r2 = 1 - np.nansum((yt.values[obs] - pred[obs]) ** 2) / np.nansum((yt.values[obs] - yt.values[obs].mean()) ** 2)
        out[f"RP_{nm}"] = pred; out[f"RR_{nm}"] = np.where(obs, yt.values - pred, np.nan); print(f"  residual target {nm}: out-of-fold R2 {r2:.3f} ({time.time()-t0:.0f}s)", flush=True)
    return pd.DataFrame(out)
if os.path.exists(RES_PATH) and not SMOKE: RESID = pd.read_pickle(RES_PATH); print("residual features loaded from cache", RESID.shape)
else:
    RESID = compute_resid()
    if not SMOKE: RESID.to_pickle(RES_PATH)
assert len(RESID) == ntr + len(Xct)
R_tr, R_te = RESID.iloc[:ntr].reset_index(drop=True), RESID.iloc[ntr:].reset_index(drop=True)
Xc, Xa, Xck, Xak = [pd.concat([d, R_tr], axis=1) for d in (Xc, Xa, Xck, Xak)]
Xct, Xat, Xckt, Xakt = [pd.concat([d, R_te], axis=1) for d in (Xct, Xat, Xckt, Xakt)]
print("features per member family now (core / all-families):", Xc.shape[1], Xa.shape[1])
if SMOKE: Xc, Xa, Xck, Xak, y = Xc.iloc[:8000], Xa.iloc[:8000], Xck.iloc[:8000], Xak.iloc[:8000], y[:8000]
n, nt = len(y), len(Xct); tidx = np.arange(ntr, ntr + nt) if not SMOKE else np.arange(8000, 8000 + nt)
# ---- q, and the decoded product: annuity = Q0*credit + F  (F ~ 129.5 monthly fee on ~75% of cash loans); label-free ----
q = al.AMT_ANNUITY / al.AMT_CREDIT; _cash = (al.NAME_CONTRACT_TYPE == "Cash loans").values; _qb = (al.AMT_ANNUITY - 129.5) / al.AMT_CREDIT
_q5, _qb5 = q.round(5), _qb.round(5)
FEE = ((_qb5.map(_qb5.value_counts()).values > _q5.map(_q5.value_counts()).values) & _cash).astype(int); Q0 = pd.Series(np.where(FEE, _qb, q))
KS = {"Q3": q.round(3).astype(str), "Q4": q.round(4).astype(str), "Q5": q.round(5).astype(str),
      "P3": Q0.round(3).astype(str), "P4": Q0.round(4).astype(str), "P5": Q0.round(5).astype(str), "PF4": Q0.round(4).astype(str) + "|" + pd.Series(FEE).astype(str)}
M0 = {"Q3": 100., "Q4": 30., "Q5": 30., "P3": 100., "P4": 30., "P5": 30., "PF4": 30.}
if SMOKE: KS = {k: v.iloc[list(range(8000)) + list(range(ntr, ntr + nt))].reset_index(drop=True) for k, v in KS.items()}
print("decoded products:", Q0.round(5).nunique(), "(q5 distinct:", q.round(5).nunique(), ") | fee share of cash loans:", round(float(FEE[_cash].mean()), 3), "| features core/all:", Xc.shape[1], Xa.shape[1])
def enc(fit, qi, keys=None):
    pr = float(y[fit].mean()); out = {}
    for k in (keys or KS):
        key = KS[k]; g = pd.DataFrame({"g": key.iloc[fit].values, "y": y[fit]}).groupby("g")["y"].agg(["sum", "count"]); s = key.iloc[qi]
        cnt = s.map(g["count"]).fillna(0).to_numpy(float); sm = s.map(g["sum"]).fillna(0).to_numpy(float)
        out[f"TE_{k}"] = (sm + M0[k] * pr) / (cnt + M0[k]); out[f"CNT_{k}"] = cnt
    return pd.DataFrame(out)
def train_enc(a):     # nested out-of-fold encodings for training rows (a row's own label never enters its own feature)
    cols = enc(a[:10], a[:5]).columns; inner = np.zeros((len(a), len(cols)))
    for ia, ib in StratifiedKFold(5, shuffle=True, random_state=7).split(a, y[a]): inner[ib] = enc(a[ia], a[ib]).values
    return pd.DataFrame(inner, columns=cols)
def onehot(Xa_, Xt_):
    z = pd.concat([Xa_, Xt_], ignore_index=True); oh = pd.get_dummies(z[CATS].astype("category"), dummy_na=True, dtype=np.float32)
    oh.columns = [f"OH_{i}" for i in range(oh.shape[1])]; Z = pd.concat([z.drop(columns=CATS), oh], axis=1)
    return Z.iloc[:len(Xa_)].reset_index(drop=True), Z.iloc[len(Xa_):].reset_index(drop=True)
Xco, Xcot = onehot(Xc, Xct); Xao, Xaot = onehot(Xa, Xat); Xaok, Xaokt = onehot(Xak, Xakt)
MONO_K = [-1 if c in ("EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3") else 0 for c in Xck.columns] + [0] * (2 * len(KS))   # + TE/CNT columns of every key
BASE = dict(objective="binary", bagging_freq=1, verbose=-1, num_threads=THREADS, seed=42)

# ---- the five members of the submitted blend. Rounds were fixed a priori (never tuned on held-out labels). name: (train feats, test feats, params, rounds fold models, rounds full-data models, full-data seeds, blend weight)
ROUND_SCALE = 1.0     # keep the a-priori rounds (evidence: more rounds with extra_trees were worse); change only to explore
MEMBERS = {
 "qf_shallow":   (Xco,  Xcot,  dict(extra_trees=True, learning_rate=0.05, num_leaves=7, max_depth=3, min_child_samples=150, feature_fraction=0.5, bagging_fraction=0.8, lambda_l2=20), 1300, 1430, [42, 7, 2024, 11, 99], 0.24),
 "qf_reg":       (Xco,  Xcot,  dict(extra_trees=True, learning_rate=0.03, num_leaves=11, min_child_samples=300, feature_fraction=0.4, bagging_fraction=0.8, lambda_l2=40), 1939, 2133, [42, 7, 2024, 11, 99], 0.12),
 "v1_allfam_oh": (Xao,  Xaot,  dict(extra_trees=True, learning_rate=0.04, num_leaves=15, min_child_samples=150, feature_fraction=0.3, bagging_fraction=0.8, lambda_l2=20), 881, 969, [42, 7, 2024], 0.32),
 "v2_allfam_oh": (Xaok, Xaokt, dict(extra_trees=True, learning_rate=0.04, num_leaves=15, min_child_samples=150, feature_fraction=0.3, bagging_fraction=0.8, lambda_l2=20), 881, 969, [42, 7, 2024], 0.16),
 "v2_mono":      (Xck.reset_index(drop=True), Xckt.reset_index(drop=True), dict(extra_trees=True, learning_rate=0.02, num_leaves=15, min_child_samples=150, feature_fraction=0.5, bagging_fraction=0.8, lambda_l2=20, monotone_constraints=MONO_K), 1571, 1729, [42, 7, 2024], 0.16)}
assert abs(sum(m[-1] for m in MEMBERS.values()) - 1) < 1e-9
rk = lambda v: (rankdata(v) - 1) / (len(v) - 1)
FOLDS = 2 if SMOKE else 5
def run_member(name):
    X, Xt, p, R, Rf, seeds, _ = MEMBERS[name]; f = os.path.join(CACHE_DIR, name + "_et.npz")
    if os.path.exists(f) and not SMOKE:
        z = np.load(f); print(f"{name}: loaded from cache | held-out AUC {roc_auc_score(y, z['oof']):.5f}"); return z["test"], z["oof"]
    tf = np.zeros(nt); oof = np.zeros(n); t0 = time.time()
    for k, (a, b) in enumerate(StratifiedKFold(FOLDS, shuffle=True, random_state=42).split(np.zeros(n), y)):
        fa = pd.concat([X.iloc[a].reset_index(drop=True), train_enc(a)], axis=1); fb = pd.concat([X.iloc[b].reset_index(drop=True), enc(a, b)], axis=1); ft = pd.concat([Xt.reset_index(drop=True), enc(a, tidx)], axis=1)
        m = lgb.train({**BASE, **p}, lgb.Dataset(fa, y[a]), 60 if SMOKE else int(R * ROUND_SCALE)); oof[b] = m.predict(fb); tf += m.predict(ft) / FOLDS
        print(f"  {name} fold {k+1}/{FOLDS}: AUC {roc_auc_score(y[b], oof[b]):.5f} ({time.time()-t0:.0f}s)", flush=True)
    allr = np.arange(n); fa = pd.concat([X.reset_index(drop=True), train_enc(allr)], axis=1); ft = pd.concat([Xt.reset_index(drop=True), enc(allr, tidx)], axis=1); fr = []
    for sd in ([42] if SMOKE else seeds):
        fr.append(rk(lgb.train({**BASE, **p, "seed": sd}, lgb.Dataset(fa, y), 60 if SMOKE else int(Rf * ROUND_SCALE)).predict(ft))); print(f"  {name} full-data seed {sd} done ({time.time()-t0:.0f}s)", flush=True)
    test = rk(0.5 * rk(tf) + 0.5 * np.mean(fr, 0)); print(f"{name}: held-out AUC {roc_auc_score(y, oof):.5f}", flush=True)
    if not SMOKE: np.savez(f, test=test, oof=oof)
    return test, oof
RES = {}

In [ ]:
# cell 3a - member 1: qf_shallow (about 12 min)
RES["qf_shallow"] = run_member("qf_shallow")

In [ ]:
# cell 3b - member 2: qf_reg (about 20 min)
RES["qf_reg"] = run_member("qf_reg")

In [ ]:
# cell 3c - member 3: v1_allfam_oh (about 7 min)
RES["v1_allfam_oh"] = run_member("v1_allfam_oh")

In [ ]:
# cell 3d - member 4: v2_allfam_oh (about 7 min)
RES["v2_allfam_oh"] = run_member("v2_allfam_oh")

In [ ]:
# cell 3e - member 5: v2_mono (about 10 min)
RES["v2_mono"] = run_member("v2_mono")

In [ ]:
# cell 4 - apples-to-apples comparison with the old members (same weights), then files   [a gain of +0.0005 or more here would be the first real step since the product decode]
from sklearn.model_selection import StratifiedKFold as SKF
W = {k: MEMBERS[k][-1] for k in MEMBERS}
old = {}
if not SMOKE:
    for k in MEMBERS:
        f = os.path.join(V6_CACHE, k + ".npz")
        assert os.path.exists(f), f"v6 member cache missing: {f} (run the v6 notebook first)"
        z = np.load(f); old[k] = (rk(z["oof"]), z["test"])
new = {k: (rk(RES[k][1]), RES[k][0]) for k in MEMBERS}
if SMOKE: old = new
blend = lambda D, i=slice(None): sum(W[k] * D[k][0][i] for k in W)
print("member held-out AUC  (v6 -> v7 extra_trees):")
for k in MEMBERS: print(f"  {k:14s} {roc_auc_score(y, old[k][0]):.5f} -> {roc_auc_score(y, new[k][0]):.5f}")
mix = lambda i=slice(None): 0.5 * blend(old, i) + 0.5 * blend(new, i)
rows = []
for f, (a, b) in enumerate(SKF(FOLDS, shuffle=True, random_state=42).split(np.zeros(n), y)):
    ao, an, am = roc_auc_score(y[b], blend(old, b)), roc_auc_score(y[b], blend(new, b)), roc_auc_score(y[b], mix(b)); rows.append((ao, an, am)); print(f"  fold {f+1}: v6 blend {ao:.5f} | v7 blend {an:.5f} ({an-ao:+.5f}) | 50/50 v6+v7 {am:.5f} ({am-ao:+.5f})")
R = np.array(rows); dn, dm = R[:, 1] - R[:, 0], R[:, 2] - R[:, 0]
print(f"\nv7 minus v6: mean {dn.mean():+.5f}, folds positive {int((dn>0).sum())}/{FOLDS} | 50/50 mix minus old: mean {dm.mean():+.5f}, folds positive {int((dm>0).sum())}/{FOLDS}")
print(f"pooled held-out AUC: v6 {roc_auc_score(y, blend(old)):.5f} | v7 {roc_auc_score(y, blend(new)):.5f} | mix {roc_auc_score(y, mix()):.5f}")
ids = sample.SK_ID_CURR
def write(name, vals):
    out = pd.DataFrame({"SK_ID_CURR": ids, "TARGET": vals}); assert len(out) == 61500 and out.SK_ID_CURR.is_unique and np.isfinite(out.TARGET).all()
    p = os.path.join(OUT_DIR, name); out.to_csv(p, index=False); print("wrote", p); return p
if not SMOKE:
    files = [write("v7_A_et_members_same_weights.csv", blend(new, slice(None)) if False else sum(W[k] * new[k][1] for k in W)), write("v7_B_mix_v6_and_et.csv", 0.5 * sum(W[k] * old[k][1] for k in W) + 0.5 * sum(W[k] * new[k][1] for k in W))]
    print("\nRule set in advance: submit a v7 file only if its 'minus old' mean is >= +0.0002 AND positive on >= 4 of 5 folds. Otherwise keep v6_B (0.778).")
    try:
        from google.colab import files as cf
        for p in files: cf.download(p)
    except Exception: print("download manually from", OUT_DIR)
else: print("SMOKE run: files not written")